# PoliticalDemocracy: linear SEM versus lavaan

Fit the bundled PoliticalDemocracy observations with `SEM2Linear` and the package's `fit()` interface. The [lavaan SEM tutorial](https://lavaan.ugent.be/tutorial/sem.html) supplies rounded, unstandardized maximum-likelihood reference coefficients—not known causal effects. This is an experiment: it reports differences rather than asserting a tolerance.

Install SEM2 from this checkout with `python -m pip install -e .` and use that environment as the notebook kernel (install `ipykernel` if needed). Open this notebook in the repository's `docs/` directory or run it from the repository root. The equivalent command-line example is:

```sh
python -m experiments.political_democracy --epochs 2000 --seed 0
```

In [ ]:
from pathlib import Path
import sys

# The examples and bundled data live in the source checkout, not the wheel.
root = Path.cwd()
if not (root / 'experiments' / 'political_democracy.py').is_file():
    root = root.parent
if not (root / 'experiments' / 'political_democracy.py').is_file():
    raise RuntimeError('Run this notebook from the checkout root or its docs directory.')
sys.path.insert(0, str(root))

import torch
from experiments.political_democracy import FORMULA, load_data, run_example

torch.set_num_threads(1)

## Model and likelihood

The latent variables are industrialization in 1960 (`ind60`) and democracy in 1960 and 1965 (`dem60`, `dem65`). Their indicators are `x1`–`x3`, `y1`–`y4`, and `y5`–`y8`, respectively. The three structural regressions are `dem60 <- ind60` and `dem65 <- ind60 + dem60`. Only the indicator columns enter fitting.

Center each observed column without standardizing its variance. All model intercepts are fixed at zero; this fits covariance structure, not the original observed means. `LogLikelihood(centered=True)` evaluates the analytical Gaussian covariance, without Monte Carlo draws. `SEM2Loss([(likelihood, -1.)])` supplies the minimizing objective to `fit()`. Each epoch is one full-data Adam update. The empirical covariance comparison uses divisor **n**, matching Gaussian ML rather than the usual unbiased divisor n−1.

Independent standard-normal noises shared by indicator pairs represent the six residual covariances in lavaan's example. For instance, `e15` enters `y1` and `y5`: the product of its two loadings contributes their covariance, and each squared loading also contributes its indicator's variance. Additional independent noises supply nonnegative residual variance. This parameterization is not guaranteed to represent every positive-definite residual covariance allowed by lavaan, and the shared-noise decomposition need not be unique.

The setup reuses the model from the [original linear SEM notebook](../notebooks/linear_sem.ipynb), with explicit zero intercepts for centered fitting.

In [ ]:
data = load_data()
print(f'{len(data)} observations, {len(data.columns)} indicators')
print(FORMULA.replace(';', ';\n'))
data.head()

In [ ]:
coefficients, covariances, diagnostics = run_example(epochs=2000, seed=0, lr=0.05)
coefficients

## Comparing coefficients

lavaan fixes each latent's first indicator loading to one. SEM2 learns those loadings, so raw structural coefficients are not directly comparable. For a path from one latent to another, multiply its coefficient by the outcome latent's first loading and divide by the predictor latent's first loading. The table applies this transformation using `x1`, `y1`, and `y5`. Unit exogenous noise variances do **not** make every latent variance one.

The lavaan reference values are 1.483, 0.572, and 0.837. They are rounded estimates from that tutorial, not a new R fit executed by this notebook. Differences can reflect incomplete optimization and parameterization restrictions; closeness alone does not establish model equivalence or causal identification.

In [ ]:
for name, value in diagnostics.items():
    print(f'{name}: {value:.6g}')

# Largest observed covariance residuals; diagonal entries are variances.
covariances.loc[covariances['difference'].abs().sort_values(ascending=False).index].head(10)

The negative log-likelihood and gradient are evaluated on the full data **after** the final update. Covariance residuals describe disagreement with the empirical moments; they need not be zero in a constrained model. The gradient depends on this parameterization and is a diagnostic, not a general convergence certificate.

Finishing 2,000 updates is budget completion, not proof of convergence. To investigate numerical sensitivity, rerun explicitly with a larger epoch budget or a different seed and compare the tables. No fitting run is launched by pytest, and there is no coefficient pass/fail threshold.